# TPE conjunto A5b/A5 (BCI IV-4): Colab desde VS Code

Optimización conjunta con **Optuna TPE** de A5b y A5 (arquitectura, `kernel_size`, `encoder_levels`, `base_filters`, LR y L2) sobre el pipeline **P0** de las ablaciones. Especificación: `docs/plan_implementacion_TPE_ECoG.md`.

- **Búsqueda:** un estudio por sujeto, **32 trials COMPLETE** × 5 folds; los 10 primeros son los warmups equilibrados del plan (§6.1).
- **Entrenamiento:** Adam + L2, early stopping sobre el Pearson de validación (`min_delta=0`, paciencia 10 desde la época 16, máximo 50) y mejor checkpoint.
- **Verificación:** mejor trial de A5b y de A5 por sujeto × 2 semillas nuevas × 5 folds.
- **Retardo:** ECoG(t) → guante(t + **20 ms**) = 2 columnas a 100 Hz, como en las ablaciones.
- No se guardan pesos: cada fold conserva métricas y `predictions.npz` del mejor checkpoint.

Este notebook se abre en **VS Code** y se ejecuta en un **servidor de Colab** con la extensión *Google Colab*: las celdas corren en la máquina remota (`/content/...`), no en tu PC.

**Preparación (una vez)**
1. Instala la extensión *Google Colab* en VS Code e inicia sesión con tu cuenta de Google.
2. Los `.mat` pueden ser los mismos de las ablaciones (`DATA_DIR`).
3. Haz `git push` de este repositorio a GitHub, idealmente con una etiqueta fija para la campaña (`GIT_REF`).

**Importante**
- **El código viene del commit clonado en el servidor**, no de tu carpeta local. Si cambias `src/`, haz commit y push y reejecuta la celda 3.
- Mantén VS Code conectado. Si el servidor se recicla, los resultados están en Drive: abre un servidor nuevo, ejecuta las celdas 1–3 y vuelve a lanzar la 7 (o la 10). El estudio se reanuda desde `results/studies/<S>_study.zip`; el trial que estaba en curso se repite con los mismos parámetros.
- **Un runtime por sujeto** si quieres paralelizar: cada servidor con `SUBJECTS` disjuntos sobre el mismo `RESULTS_DIR`. Nunca dos procesos sobre el mismo sujeto.
- Usa la **misma clase de GPU** en toda la campaña: TF32 cambia ligeramente la aritmética entre GPUs.

## 1. GPU y Google Drive
Si `drive.mount` no lanza el inicio de sesión, usa el comando de VS Code *Colab: Mount Google Drive to Server...*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 2. Configuración

In [ ]:
GIT_URL = "https://github.com/SaulRodas/TPE_ECoG.git"   # ← tu repositorio ("" = código subido a mano)
GIT_REF = "main"           # recomendado: una etiqueta fija para toda la campaña, p. ej. "tpe-v1"
PRIVATE_REPO = False       # True → la celda 3 pide un token de GitHub de solo lectura
REPO_DIR = "/content/TPE_ECoG"   # carpeta del código EN EL SERVIDOR

DRIVE_ROOT = "/content/drive/MyDrive"
DATA_DIR = f"{DRIVE_ROOT}/ECoG_Architectures_Exploration/data/raw"   # mismos .mat que las ablaciones
RESULTS_DIR = f"{DRIVE_ROOT}/TPE_ECoG/results"
WORK_DIR = "/content/tpe_work"     # disco local del servidor: SQLite de trabajo (se publica en Drive tras cada trial)

SUBJECTS = ["S1", "S2", "S3"]      # un runtime por sujeto para paralelizar, p. ej. ["S2"]

import os
missing = [f for f in ("sub1_comp.mat", "sub2_comp.mat", "sub3_comp.mat") if not os.path.exists(f"{DATA_DIR}/{f}")]
print("datos OK" if not missing else f"faltan en {DATA_DIR}: {missing}")
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Código y dependencias en el servidor
- **Opción A (recomendada):** se clona `GIT_URL` en `GIT_REF`; el commit queda registrado en cada trial.
- **Opción B:** con `GIT_URL = ""`, sube `src`, `scripts`, `configs`, `tests`, `reference` y `pyproject.toml` con clic derecho → *Upload to Colab* y pon `REPO_DIR = "/content"`.

Solo se instalan `mne` y `optuna==5.0.0` (lo demás viene en Colab).

In [ ]:
import base64, getpass, subprocess, sys

TOKEN = getpass.getpass("Token de GitHub (solo lectura): ") if (GIT_URL and PRIVATE_REPO) else None

def git(*args, cwd=None):
    cmd = ["git"]
    if TOKEN:
        basic = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
        cmd += ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
    out = subprocess.run(cmd + list(args), cwd=cwd, capture_output=True, text=True)
    if out.returncode != 0:
        message = out.stderr.replace(TOKEN, "***") if TOKEN else out.stderr
        raise RuntimeError(f"git {args[0]} falló: {message[-800:]}")
    return out.stdout.strip()

if GIT_URL:
    if not os.path.exists(f"{REPO_DIR}/.git"):
        git("clone", "--quiet", GIT_URL, REPO_DIR)
    git("fetch", "--quiet", "--tags", "origin", cwd=REPO_DIR)
    git("checkout", "--quiet", GIT_REF, cwd=REPO_DIR)
    if git("rev-parse", "--abbrev-ref", "HEAD", cwd=REPO_DIR) != "HEAD":   # rama (no etiqueta): actualizar
        git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
    print("commit:", git("log", "-1", "--format=%h %s (%ci)", cwd=REPO_DIR))
else:
    assert os.path.exists(f"{REPO_DIR}/src/ecog_tpe"), f"no está el código en {REPO_DIR} (¿Upload to Colab?)"
    print("usando código subido en", REPO_DIR)

!pip install -q "mne>=1.6" "optuna==5.0.0"

sys.path.insert(0, f"{REPO_DIR}/src")
from ecog_tpe.config import load_config
from ecog_tpe import reporting
CFG = load_config(f"{REPO_DIR}/configs/tpe.yaml")
print("configuración:", CFG["_path"])
print("retardo", CFG["preprocessing"]["delay_ms"], "ms =", CFG["_derived"]["delay_columns"], "columnas | TF32",
      CFG["runtime"]["allow_tf32"], "| early stopping", CFG["training"]["early_stopping"])

## 4. Comprobaciones mínimas (`pytest`)
Pipeline P0 (datos reales, fugas, retardo de 2 columnas, DSP frente a MNE), las 16 combinaciones de A5b/A5, la política de parada (incluida la tabla 1.1 del plan sobre los históricos de las ablaciones), `train_fold` y el estudio Optuna con un runner simulado (warmups, reanudación, fallos, verificación y reportes).

In [ ]:
!cd "{REPO_DIR}" && ECOG_DATA_DIR="{DATA_DIR}" python -m pytest -q -p no:cacheprovider 2>&1 | tail -5

## 5. Capacidad de la GPU (plan §3.2)
Forward/backward + paso de Adam con batch 128 en las **16 combinaciones** discretas, con el sujeto de más canales (S3) y un tensor del tamaño de X del fold en la GPU. **Si alguna da OOM, hay que ajustar el espacio antes del primer estudio** (no se permite cambiar batch ni precisión por trial). La estimación de horas usa 32.3 épocas por fold (media retrospectiva) y es solo orientativa.

In [ ]:
!cd "{REPO_DIR}" && python scripts/run_tpe.py --capacity --results-dir "{RESULTS_DIR}" 2>&1 | grep -v Warning | tail -24

## 6. Smoke real (S1/B1, warmups del par 1, 2 épocas × 2 batches)
Recorre la cadena completa con datos reales en `RESULTS_DIR/_smoke` (separado de la campaña; se borra en cada smoke): características P0, estudio, warmups, objetivo, publicación del zip y reporte.

In [ ]:
!cd "{REPO_DIR}" && python scripts/run_tpe.py --smoke --results-dir "{RESULTS_DIR}" --work-dir "{WORK_DIR}" 2>&1 | grep -v Warning | tail -20

import glob, json
import pandas as pd
from IPython.display import display
rows = []
for path in sorted(glob.glob(f"{RESULTS_DIR}/_smoke/trials/S1/trial_*/B1/history.csv")):
    h = pd.read_csv(path)
    config = json.load(open(path.replace("B1/history.csv", "config.json"), encoding="utf-8"))
    rows.append({"trial": path.split("/")[-3], "arquitectura": config["label"],
                 "s_por_batch": (h.train_seconds / h.n_batches).iloc[-1], "s_validación": h.val_seconds.iloc[-1]})
display(pd.DataFrame(rows).round(4))
!cd "{REPO_DIR}" && python scripts/report_tpe.py --results-dir "{RESULTS_DIR}/_smoke" 2>&1 | tail -12

## 7. Lanzar la búsqueda en segundo plano
Un proceso independiente recorre `SUBJECTS` en orden, cada uno hasta 32 trials COMPLETE (reanudable). El log queda en `RESULTS_DIR/logs/`. Si ya hay un runner vivo en este servidor, el nuevo se niega a arrancar.

In [ ]:
import subprocess, sys, time
os.makedirs(f"{RESULTS_DIR}/logs", exist_ok=True)
LOG_PATH = f"{RESULTS_DIR}/logs/tpe_{'_'.join(SUBJECTS)}_{time.strftime('%Y%m%d_%H%M%S')}.log"
args = [sys.executable, "-u", f"{REPO_DIR}/scripts/run_tpe.py", "--subjects", *SUBJECTS,
        "--data-dir", DATA_DIR, "--results-dir", RESULTS_DIR, "--work-dir", WORK_DIR]
PROC = subprocess.Popen(args, stdout=open(LOG_PATH, "a"), stderr=subprocess.STDOUT, cwd=REPO_DIR,
                        start_new_session=True)
time.sleep(30)
print("PID", PROC.pid, "| estado:", "en marcha" if PROC.poll() is None else f"terminó (código {PROC.returncode})")
print("log:", LOG_PATH, "  (en la terminal de Colab: tail -f", LOG_PATH + ")")
print(open(LOG_PATH).read()[-2000:])

## 8. Monitor (reejecutable)
Progreso por sujeto (COMPLETE/FAIL de 32, mejor objetivo por arquitectura, ETA), trial y fold en curso con su curva en vivo, latido del runner y últimas líneas del log. **Detener esta celda (■) no detiene el entrenamiento.**

In [ ]:
reporting.monitor(RESULTS_DIR, CFG, SUBJECTS, refresh=30)

## 9. Detener
Envía SIGINT al runner de este servidor: el trial en curso no se publica y se repetirá (con los mismos parámetros) al relanzar la celda 7 o la 10.

In [ ]:
import glob, json, signal, socket
for path in glob.glob(f"{RESULTS_DIR}/runners/{socket.gethostname()}-*.json"):
    pid = json.load(open(path))["pid"]
    try:
        os.killpg(os.getpgid(pid), signal.SIGINT)
        print("SIGINT enviado al runner", pid)
    except ProcessLookupError:
        print("el runner", pid, "ya no existe")

## 10. Verificación de finalistas (plan §7)
Cuando un sujeto tiene **32 trials COMPLETE**: su mejor trial de A5b y de A5 se reentrenan con las semillas 1042 y 2042 (+ 5·sujeto + fold) en los cinco folds, con la misma política. La elección queda en `verification/<S>/finalists.json` y no se recalcula al reanudar. Se detiene con la sección 9.

In [ ]:
import json, zipfile
for s in SUBJECTS:
    path = f"{RESULTS_DIR}/studies/{s}_study.zip"
    meta = json.loads(zipfile.ZipFile(path).read("meta.json")) if os.path.exists(path) else {}
    print(s, "COMPLETE:", meta.get("n_complete", 0), "/", CFG["search"]["n_trials_per_subject"])

LOG_PATH = f"{RESULTS_DIR}/logs/verify_{'_'.join(SUBJECTS)}_{time.strftime('%Y%m%d_%H%M%S')}.log"
args = [sys.executable, "-u", f"{REPO_DIR}/scripts/run_tpe.py", "--subjects", *SUBJECTS, "--verify",
        "--data-dir", DATA_DIR, "--results-dir", RESULTS_DIR]
PROC = subprocess.Popen(args, stdout=open(LOG_PATH, "a"), stderr=subprocess.STDOUT, cwd=REPO_DIR,
                        start_new_session=True)
time.sleep(30)
print("PID", PROC.pid, "| estado:", "en marcha" if PROC.poll() is None else f"terminó (código {PROC.returncode})")
print(open(LOG_PATH).read()[-2000:])

In [ ]:
# Monitor de la verificación (reejecutable; detenerlo no detiene el entrenamiento)
reporting.monitor(RESULTS_DIR, CFG, SUBJECTS, refresh=30, verification=True)

## 11. Reporte
Genera `RESULTS_DIR/summary/` con lo que haya (se puede ejecutar a mitad de campaña): `tpe_trials.csv`, `tpe_folds.csv`, `tpe_verification.csv`, `tpe_selected_models.csv`, la retrospectiva de las ablaciones (`ablation_retrospective.csv`, tabla 1.1) y la comparación del §10 (`tpe_comparison.csv`, por sujeto, dedo y fold). Recuerda: los mejores checkpoints son métricas de **selección sobre validación**, no rendimiento de test.

In [ ]:
!cd "{REPO_DIR}" && python scripts/report_tpe.py --results-dir "{RESULTS_DIR}" 2>&1 | tail -40

import pandas as pd
from IPython.display import display
S = f"{RESULTS_DIR}/summary"
trials = pd.read_csv(f"{S}/tpe_trials.csv")
display(trials[trials.state == "COMPLETE"].sort_values("objective_cv_best", ascending=False)
        .groupby("subject").head(5)[["subject", "trial", "source", "label", "learning_rate", "weight_decay",
                                     "objective_cv_best", "mean_best_epoch", "mean_epochs_trained"]].round(5))
reporting.show_images([f"{S}/figures/{n}.png" for n in
                       ("optimization_history", "params_vs_objective", "reference_curves", "verification")])

## 12. Al terminar la sesión
Comprueba antes que no queda ningún runner vivo (sección 8) y después fuerza la sincronización de Drive antes de liberar el servidor (*Colab: Remove Server*).

In [ ]:
try:
    drive.flush_and_unmount()
    print("Drive sincronizado y desmontado")
except Exception as exc:  # p. ej. si Drive se montó desde el comando de la extensión
    print("no se pudo desmontar Drive:", exc)